<a href="https://colab.research.google.com/github/emilheroyt/Bachelor-Thesis-Stress-Testing-Robustness-of-SOTA-Point-Trackers/blob/main/CoWTrackerIsolation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zelle 1 – GPU-Check:



In [ ]:
!nvidia-smi

Wed Sep  2 11:25:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   37C    P0             58W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Zelle 2 – Repo klonen:

In [ ]:
%cd /content
!rm -rf cowtracker main.zip cowtracker-main
!wget -q https://github.com/facebookresearch/cowtracker/archive/refs/heads/main.zip
!unzip -q main.zip
!mv cowtracker-main cowtracker
%cd /content/cowtracker
!ls

/content
/content/cowtracker
CODE_OF_CONDUCT.md  cowtracker	docs		  LICENSE    videos
CONTRIBUTING.md     demo.py	environments.yml  README.md


In [ ]:
%cd /content/cowtracker/cowtracker/thirdparty
!rm -rf DepthAnythingV2 vggt
!wget -q https://github.com/DepthAnything/Depth-Anything-V2/archive/refs/heads/main.zip -O da2.zip
!unzip -q da2.zip && mv Depth-Anything-V2-main DepthAnythingV2

!wget -q https://github.com/facebookresearch/vggt/archive/refs/heads/main.zip -O vggt.zip
!unzip -q vggt.zip && mv vggt-main vggt

%cd /content/cowtracker
!ls cowtracker/thirdparty/

/content/cowtracker/cowtracker/thirdparty
/content/cowtracker
da2.zip  DepthAnythingV2  __init__.py  vggt  vggt.zip


Zelle 3 – Fehlende Pakete:

In [ ]:
!pip install mediapy xformers -q

Zelle 4 – FlashAttention3-Patch:

In [ ]:
!ls /content/

cowtracker  main.zip  sample_data


In [ ]:
import re

path = "/content/cowtracker/cowtracker/layers/video_transformer.py"
with open(path, "r") as f:
    content = f.read()

pattern = r"(def forward\(\s*self, x: torch\.Tensor, attn_mask: torch\.Tensor \| None = None)(\s*\) -> torch\.Tensor:)"
replacement = r"\1, is_causal: bool = False, **kwargs\2"

new_content, n = re.subn(pattern, replacement, content)
print(f"{n} Stelle(n) ersetzt")

with open(path, "w") as f:
    f.write(new_content)

1 Stelle(n) ersetzt


Zelle 5 – Demo laufen lassen (erst moderat, z. B. 50 Frames):

In [ ]:
!python demo.py --video videos/bmx-bumps.mp4 --output output.mp4 --max_frames 50

timm version:  1.0.28
CoWTracker Inference Demo

[1/4] Loading model...
Initializing CoWTracker...
✓ Flash Attention 3 enabled for spatial attention: replaced 12 attention modules
CowTrackingHead initialized: iter_dim=64, warp_iters=5
  - Features: 128, Side channels: 128
  - Warping-based iterative refinement iterations: 5

cowtracker_model.pth: downloading bytes:   0% 7.33M/3.90G [00:01<11:27, 5.66MB/s]
cowtracker_model.pth: downloading bytes:   0% 15.4M/3.90G [00:01<04:55, 13.1MB/s,  712kB/s  ]
cowtracker_model.pth: downloading bytes:   1% 49.1M/3.90G [00:01<01:12, 53.2MB/s, 3.03MB/s  ]
cowtracker_model.pth: downloading bytes:   2% 74.1M/3.90G [00:01<00:43, 88.0MB/s, 4.69MB/s  ]
cowtracker_model.pth: downloading bytes:   3% 104M/3.90G [00:01<00:29, 129MB/s, 7.05MB/s  ]  
cowtracker_model.pth: downloading bytes:   3% 127M/3.90G [00:01<00:24, 152MB/s, 9.80MB/s  ]
cowtracker_model.pth: downloading bytes:   9% 334M/3.90G [00:02<00:15, 237MB/s, 27.0MB/s  ]
cowtracker_model.pth: downloadi

Zelle 6 – Ergebnis anschauen:

In [ ]:
from IPython.display import Video
Video("output.mp4", embed=True)

Zelle 1 – TAP-Vid-DAVIS in diesem Notebook laden (falls noch nicht geschehen) + dasselbe Video wie bei TapNext (Index 1, mit echter Occlusion):

In [ ]:
!wget -q https://github.com/google-deepmind/tapnet/archive/refs/heads/main.zip -O tapnet_repo.zip
!unzip -q -o tapnet_repo.zip
!pip install "./tapnet-main[torch]" -q

!rm -f tapvid_davis.zip
!wget https://storage.googleapis.com/dm-tapnet/tapvid_davis.zip
!ls -la tapvid_davis.zip
!unzip -q -o tapvid_davis.zip
!ls tapvid_davis/

# Git-Credential-Problem beheben (für den pip install weiter unten nötig)
!git config --global credential.helper ""
import os
os.environ["GIT_TERMINAL_PROMPT"] = "0"

# evaluation_datasets.py laden – das fehlte in deiner bisherigen Zelle
!wget -q https://raw.githubusercontent.com/google-deepmind/tapnet/refs/heads/main/tapnet/tapvid/evaluation_datasets.py

!pip install "tapnet[torch] @ git+https://github.com/google-deepmind/tapnet.git" -q

import sys
from unittest.mock import MagicMock

sys.modules['tensorflow'] = MagicMock()
sys.modules['tensorflow_datasets'] = MagicMock()
sys.modules['tensorflow'].io.gfile.GFile = open

sys.path.insert(0, '.')
import evaluation_datasets
from evaluation_datasets import create_davis_dataset, compute_tapvid_metrics
import numpy as np

davis = create_davis_dataset('tapvid_davis/tapvid_davis.pkl', query_mode='first')
for i, sample in enumerate(davis):
    if i == 1:
        batch = sample['davis']
        break

print(batch['video'].shape, batch['video'].min(), batch['video'].max())
print(batch['query_points'].shape)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.1.0 which is incompatible.
--2026-09-02 11:42:20--  https://storage.googleapis.com/dm-tapnet/tapvid_da

CoWTracker Pretrain laden

In [ ]:
import torch.nn.functional as F

def apply_motion_blur(frames, center_idx, window_size, gamma=2.2):
    half = window_size // 2
    start = max(0, center_idx - half)
    end = min(len(frames), center_idx + half + 1)
    window = frames[start:end].astype(np.float32) / 255.0
    linear = window ** gamma
    linear_avg = linear.mean(axis=0)
    blurred = linear_avg ** (1.0 / gamma)
    return (blurred * 255).astype(np.uint8)

def blur_video(frames, window_size):
    return np.stack([apply_motion_blur(frames, i, window_size) for i in range(len(frames))])

BLUR_LEVELS = {"none": 1, "medium": 3, "strong": 7}

def pad_to_112(video_tensor):
    patch_size = 112
    H, W = video_tensor.shape[-2:]
    pad_h = (patch_size - H % patch_size) % patch_size
    pad_w = (patch_size - W % patch_size) % patch_size
    return F.pad(video_tensor, (0, pad_w, 0, pad_h))

def track_query_points_cowtracker(model, video_chw, query_points):
    T = video_chw.shape[0]
    N = len(query_points)
    pred_tracks = np.zeros((N, T, 2), dtype=np.float32)
    pred_vis = np.zeros((N, T), dtype=bool)

    unique_times = np.unique(query_points[:, 0]).astype(int)

    for qt in unique_times:
        idx_mask = query_points[:, 0].astype(int) == qt
        pts = query_points[idx_mask]
        ys = np.round(pts[:, 1]).astype(int).clip(0, video_chw.shape[-2] - 1)
        xs = np.round(pts[:, 2]).astype(int).clip(0, video_chw.shape[-1] - 1)
        global_indices = np.where(idx_mask)[0]

        fwd_padded = pad_to_112(video_chw[qt:]).half()
        with torch.no_grad():
            pred_fwd = model(fwd_padded)
        track_fwd, vis_fwd = pred_fwd['track'][0], pred_fwd['vis'][0]

        for gi, y, x in zip(global_indices, ys, xs):
            pred_tracks[gi, qt:] = track_fwd[:, y, x].float().cpu().numpy()
            pred_vis[gi, qt:] = vis_fwd[:, y, x].float().cpu().numpy() > 0.5

        if qt > 0:
            bwd_padded = pad_to_112(video_chw[:qt + 1].flip(0)).half()
            with torch.no_grad():
                pred_bwd = model(bwd_padded)
            track_bwd, vis_bwd = pred_bwd['track'][0], pred_bwd['vis'][0]

            for gi, y, x in zip(global_indices, ys, xs):
                pred_tracks[gi, :qt + 1] = track_bwd[:, y, x].float().cpu().numpy()[::-1]
                pred_vis[gi, :qt + 1] = vis_bwd[:, y, x].float().cpu().numpy()[::-1] > 0.5

    return pred_tracks, pred_vis

query_points = batch['query_points'][0]
frames_uint8 = ((batch['video'][0] + 1) / 2 * 255).astype(np.uint8)

Kurzer 1 Video Test

In [ ]:
import sys

for mod_name in ["tensorflow", "tensorflow_datasets"]:
    if mod_name in sys.modules:
        del sys.modules[mod_name]

sys.path.insert(0, "/content/cowtracker")

for mod_name in list(sys.modules):
    if mod_name == "cowtracker" or mod_name.startswith("cowtracker."):
        del sys.modules[mod_name]

import torch
from cowtracker import CoWTracker

model = CoWTracker.from_checkpoint(device="cuda", dtype=torch.float16)

timm version:  1.0.28
Initializing CoWTracker...
✓ Flash Attention 3 enabled for spatial attention: replaced 12 attention modules
CowTrackingHead initialized: iter_dim=64, warp_iters=5
  - Features: 128, Side channels: 128
  - Warping-based iterative refinement iterations: 5


Downloaded to: /root/.cache/huggingface/hub/models--facebook--cowtracker/snapshots/860b85cfb3a63b91780535689c262b3e595b6082/cowtracker_model.pth
Detected legacy checkpoint format, remapping keys...
Load message: <All keys matched successfully>
Model loaded successfully!


In [ ]:
import torch
import time

start = time.time()
for level_name, window in BLUR_LEVELS.items():
    blurred_frames = blur_video(frames_uint8, window)
    blurred_chw = torch.from_numpy(blurred_frames).permute(0, 3, 1, 2).float().cuda()
    pred_tracks, pred_vis = track_query_points_cowtracker(model, blurred_chw, query_points)
elapsed = time.time() - start
print(f"Zeit für 1 Video (3 Blur-Stufen): {elapsed:.1f} Sekunden")
print(f"Hochgerechnet auf 30 Videos: {elapsed * 30 / 60:.1f} Minuten")

/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Zeit für 1 Video (3 Blur-Stufen): 87.0 Sekunden
Hochgerechnet auf 30 Videos: 43.5 Minuten


Vorwärts und Rückwärts Logik pro Query Zeit nachbauen:

In [ ]:
def pad_to_112(video_tensor):
    patch_size = 112
    H, W = video_tensor.shape[-2:]
    pad_h = (patch_size - H % patch_size) % patch_size
    pad_w = (patch_size - W % patch_size) % patch_size
    return F.pad(video_tensor, (0, pad_w, 0, pad_h))

def track_query_points_cowtracker(model, video_chw, query_points):
    """
    video_chw: (T, 3, H, W) float, Werte [0,255], NICHT gepolstert
    query_points: (N, 3) Array mit (t, y, x)
    """
    T = video_chw.shape[0]
    N = len(query_points)
    pred_tracks = np.zeros((N, T, 2), dtype=np.float32)
    pred_vis = np.zeros((N, T), dtype=bool)

    unique_times = np.unique(query_points[:, 0]).astype(int)

    for qt in unique_times:
        idx_mask = query_points[:, 0].astype(int) == qt
        pts = query_points[idx_mask]
        ys = np.round(pts[:, 1]).astype(int).clip(0, video_chw.shape[-2] - 1)
        xs = np.round(pts[:, 2]).astype(int).clip(0, video_chw.shape[-1] - 1)
        global_indices = np.where(idx_mask)[0]

        # Vorwärts: ab qt bis Ende
        fwd_padded = pad_to_112(video_chw[qt:]).half()
        with torch.no_grad():
            pred_fwd = model(fwd_padded)
        track_fwd, vis_fwd = pred_fwd['track'][0], pred_fwd['vis'][0]

        for gi, y, x in zip(global_indices, ys, xs):
            pred_tracks[gi, qt:] = track_fwd[:, y, x].float().cpu().numpy()
            pred_vis[gi, qt:] = vis_fwd[:, y, x].float().cpu().numpy() > 0.5

        # Rückwärts: von 0 bis qt, gespiegelt
        if qt > 0:
            bwd_padded = pad_to_112(video_chw[:qt + 1].flip(0)).half()
            with torch.no_grad():
                pred_bwd = model(bwd_padded)
            track_bwd, vis_bwd = pred_bwd['track'][0], pred_bwd['vis'][0]

            for gi, y, x in zip(global_indices, ys, xs):
                pred_tracks[gi, :qt + 1] = track_bwd[:, y, x].float().cpu().numpy()[::-1]
                pred_vis[gi, :qt + 1] = vis_bwd[:, y, x].float().cpu().numpy()[::-1] > 0.5

    return pred_tracks, pred_vis

Blur Levels Definieren

In [ ]:
def apply_motion_blur(frames, center_idx, window_size, gamma=2.2):
    half = window_size // 2
    start = max(0, center_idx - half)
    end = min(len(frames), center_idx + half + 1)
    window = frames[start:end].astype(np.float32) / 255.0
    linear = window ** gamma
    linear_avg = linear.mean(axis=0)
    blurred = linear_avg ** (1.0 / gamma)
    return (blurred * 255).astype(np.uint8)

def blur_video(frames, window_size):
    return np.stack([apply_motion_blur(frames, i, window_size) for i in range(len(frames))])

BLUR_LEVELS = {"none": 1, "medium": 3, "strong": 7}

Eigentliche Metrik-Berechnung mit Blur:

In [ ]:
results = {}

for level_name, window in BLUR_LEVELS.items():
    blurred_frames = blur_video(frames_uint8, window)  # (T, H, W, 3), uint8
    blurred_chw = torch.from_numpy(blurred_frames).permute(0, 3, 1, 2).float().cuda()

    pred_tracks, pred_vis = track_query_points_cowtracker(model, blurred_chw, query_points)

    # compute_tapvid_metrics erwartet Form (1, N, T, ...) und (t,y,x)-Query-Format
    pred_tracks_batched = pred_tracks[None]        # (1, N, T, 2), bereits (x,y)
    pred_occluded_batched = (~pred_vis)[None]      # (1, N, T)

    scalars = compute_tapvid_metrics(
        batch['query_points'], batch['occluded'], batch['target_points'],
        pred_occluded_batched, pred_tracks_batched,
        query_mode='first',
    )
    scalars = {k: float(np.sum(v)) for k, v in scalars.items()}
    results[level_name] = scalars
    print(level_name, "AJ:", scalars['average_jaccard'], "OA:", scalars['occlusion_accuracy'])

none AJ: 0.35882563590757544 OA: 0.8076079005120702
medium AJ: 0.272648605649915 OA: 0.7066569129480614
strong AJ: 0.22164102361974553 OA: 0.667885881492319


Zeitmessung für ein Video (nutzt dein schon geladenes batch von Video 1):

In [ ]:
import time

start = time.time()
for level_name, window in BLUR_LEVELS.items():
    blurred_frames = blur_video(frames_uint8, window)
    blurred_chw = torch.from_numpy(blurred_frames).permute(0, 3, 1, 2).float().cuda()
    pred_tracks, pred_vis = track_query_points_cowtracker(model, blurred_chw, query_points)
elapsed = time.time() - start
print(f"Zeit für 1 Video (3 Blur-Stufen): {elapsed:.1f} Sekunden")
print(f"Hochgerechnet auf 30 Videos: {elapsed * 30 / 60:.1f} Minuten")

Zeit für 1 Video (3 Blur-Stufen): 85.4 Sekunden
Hochgerechnet auf 30 Videos: 42.7 Minuten


Drive mounten:

In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/ba_results', exist_ok=True)

Mounted at /content/drive


der eigentliche Vollauf

In [ ]:
results_path = "/content/drive/MyDrive/ba_results/cowtracker_all_videos.csv"
all_results = []

davis_dataset_full = create_davis_dataset(
    davis_points_path='./tapvid_davis/tapvid_davis.pkl',
    query_mode='first',
    full_resolution=False,
    resolution=(256, 256),
)

for video_idx, sample in enumerate(davis_dataset_full):
    batch_i = sample['davis']
    query_points_i = batch_i['query_points'][0]
    frames_uint8_i = ((batch_i['video'][0] + 1) / 2 * 255).astype(np.uint8)

    for level_name, window in BLUR_LEVELS.items():
        blurred_frames = blur_video(frames_uint8_i, window)
        blurred_chw = torch.from_numpy(blurred_frames).permute(0, 3, 1, 2).float().cuda()

        pred_tracks, pred_vis = track_query_points_cowtracker(model, blurred_chw, query_points_i)

        pred_tracks_batched = pred_tracks[None]
        pred_occluded_batched = (~pred_vis)[None]

        scalars = compute_tapvid_metrics(
            batch_i['query_points'], batch_i['occluded'], batch_i['target_points'],
            pred_occluded_batched, pred_tracks_batched,
            query_mode='first',
        )
        scalars = {k: float(np.sum(v)) for k, v in scalars.items()}

        row = {"video": video_idx, "model": "CoWTracker", "blur": level_name}
        row.update({k: v * 100 for k, v in scalars.items()})
        all_results.append(row)

    pd.DataFrame(all_results).to_csv(results_path, index=False)
    print(f"Video {video_idx}/29 fertig")

print("Alle 30 Videos abgeschlossen.")
df = pd.DataFrame(all_results)
print(df.groupby("blur")[["average_jaccard", "occlusion_accuracy", "average_pts_within_thresh"]].mean())

Video 0/29 fertig
Video 1/29 fertig
Video 2/29 fertig
Video 3/29 fertig
Video 4/29 fertig
Video 5/29 fertig
Video 6/29 fertig
Video 7/29 fertig
Video 8/29 fertig
Video 9/29 fertig
Video 10/29 fertig
Video 11/29 fertig
Video 12/29 fertig
Video 13/29 fertig
Video 14/29 fertig
Video 15/29 fertig
Video 16/29 fertig
Video 17/29 fertig
Video 18/29 fertig
Video 19/29 fertig
Video 20/29 fertig
Video 21/29 fertig
Video 22/29 fertig
Video 23/29 fertig
Video 24/29 fertig
Video 25/29 fertig
Video 26/29 fertig
Video 27/29 fertig
Video 28/29 fertig
Video 29/29 fertig
Alle 30 Videos abgeschlossen.
        average_jaccard  occlusion_accuracy  average_pts_within_thresh
blur                                                                  
medium        48.242677           88.486367                  62.725759
none          56.898783           90.839517                  71.230006
strong        33.970111           84.666453                  46.942087
